In [1]:
import pandas as pd
import numpy as np
import os
OUTPUT_DIR = "brat_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
cases = pd.read_excel("normalized_annotations_ace_confirmed_cases_v2.xlsx")
print(len(cases.groupby('ehr_id')))
print(cases.columns)

89
Index(['ehr_id', 'concept_id', 'ACE span', 'negation', 'childhood', 'adult',
       'temporality', 'frequency', 'perpetrator', 'negated', 'childhood_cue',
       'childhood_term', 'adult_cue', 'adult_term', 'past_t_cue',
       'past_t_term', 'recent_cue', 'recent_term', 'current_cue',
       'current_term', 'ongoing_cue', 'ongoing_term', 'frequency_cue',
       'frequency_term', 'perpetrator_cue', 'perpetrator_term', 'ehr_history',
       'ace_start_char', 'ace_end_char', 'negation_start_char',
       'negation_end_char', 'childhood_start_char', 'childhood_end_char',
       'adult_start_char', 'adult_end_char', 'frequency_start_char',
       'frequency_end_char', 'past_t_start_char', 'past_t_end_char',
       'recent_start_char', 'recent_end_char', 'current_start_char',
       'current_end_char', 'ongoing_start_char', 'ongoing_end_char',
       'perpetrator_start_char', 'perpetrator_end_char'],
      dtype='str')


In [4]:
def adjust_offset(text, offset):
    """
    Adjust offset from CRLF -> LF
    Each \r\n becomes \n, so we subtract number of CRLF occurrences before offset
    """
    shift = text[:offset].count("\r\n")
    return offset - shift

def add_span(ann_lines, text, original_text, start, end, label, t_counter):
    start_adj = adjust_offset(original_text, int(start))
    end_adj = adjust_offset(original_text, int(end))

    span_text = text[start_adj:end_adj]

    t_id = f"T{t_counter}"
    ann_lines.append(f"{t_id}\t{label} {start_adj} {end_adj}\t{span_text}")

    return t_id, t_counter + 1

def to_brat_ann(df):

    for ehr_id, group in df.groupby("ehr_id"):

        original_text = group.iloc[0]["ehr_history"]
        text = original_text.replace("\r\n", "\n")

        txt_path = os.path.join(OUTPUT_DIR, f"{ehr_id}.txt")
        ann_path = os.path.join(OUTPUT_DIR, f"{ehr_id}.ann")

        with open(txt_path, "w", encoding="utf-8", newline="\n") as f:
            f.write(text)

        ann_lines = []
        t_counter = 1
        r_counter = 1
        a_counter = 1

        for _, row in group.iterrows():

            # -------------------------
            # 1. ACE entity
            # -------------------------
            ace_id, t_counter = add_span(
                ann_lines,
                text,
                original_text,
                row["ace_start_char"],
                row["ace_end_char"],
                row["concept_id"],
                t_counter
            )

            # -------------------------
            # 2. Cue spans + relations
            # -------------------------
            cue_map = [
                ("negation", "NEGATION_CUE", "Negation"),
                ("childhood", "CHILDHOOD_CUE", "Childhood"),
                ("adult", "ADULT_CUE", "Adult"),
                ("frequency", "FREQUENCY_CUE", "Frequency"),
                ("past_t", "TEMPORAL_CUE", "Temporality"),
                ("recent", "TEMPORAL_CUE", "Temporality"),
                ("current", "TEMPORAL_CUE", "Temporality"),
                ("ongoing", "TEMPORAL_CUE", "Temporality"),
                ("perpetrator", "PERPETRATOR_CUE", "Perpetrator"),
            ]

            for prefix, entity_label, relation_name in cue_map:

                start_col = f"{prefix}_start_char"
                end_col = f"{prefix}_end_char"

                if pd.notna(row[start_col]) and pd.notna(row[end_col]):

                    cue_id, t_counter = add_span(
                        ann_lines,
                        text,
                        original_text,
                        row[start_col],
                        row[end_col],
                        entity_label,
                        t_counter
                    )

                    # --- Relation (ACE → cue)
                    r_id = f"R{r_counter}"
                    ann_lines.append(f"{r_id}\t{relation_name} Arg1:{ace_id} Arg2:{cue_id}")
                    r_counter += 1

            # -------------------------
            # 3. Attributes (WITH VALUES matching the conf file)
            # -------------------------
            attr_map = {
                "negation": row["negation"],
                "childhood": row["childhood"],
                "adult": row["adult"],
                "temporality": row["temporality"],
                "frequency": row["frequency"],
                "perpetrator": row["perpetrator"],
            }

            for attr_name, value in attr_map.items():

                if pd.notna(value):
                    value_str = str(value).strip().lower()
                    # Skip default negatives
                    if value_str == "false": continue
                    a_id = f"A{a_counter}"
                    ann_lines.append(f"{a_id}\t{attr_name.capitalize()} {ace_id} {value}")
                    a_counter += 1

        with open(ann_path, "w", encoding="utf-8", newline="\n") as f:
            f.write("\n".join(ann_lines))

In [5]:
# run to generate .txt and .ann files for each EHR case
to_brat_ann(cases)